In [3]:
%pip install -U scikit-learn


  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB 1.9 MB/s eta 0:00:05
   ---------------------------------------- 0.1/8.3 MB 1.7 MB/s eta 0:00:05
   ---------------------------------------- 0.1/8.3 MB 1.7 MB/s eta 0:00:05
   ---------------------------------------- 0.1/8.3 MB 1.7 MB/s eta 0:00:05
   ---------------------------------------- 0.1/8.3 MB 1.7 MB/s eta 0:00:05
   ---------------------------------------- 0.1/8.3 MB 1.7 MB/s eta 0:00:05
    --------------------------------------- 0.1/8.3 MB 448.2 kB/s eta 0:00:19
    --------------------------------------- 0.2/8.3 MB 530.7 kB/s eta 0:00:16
   - -------------------------------------- 0.2/8.3 MB 573.4 kB/s eta 0:00:15
   - -------------------------------------- 0.3/8.3 MB 585.8 kB/s eta 0:00:14
   - ------------------------


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Semilla global
SEED = 42
rng = np.random.default_rng(SEED)

# Índices COCO-17 y parámetros temporales
L_SHOULDER, R_SHOULDER = 5, 6
L_HIP, R_HIP = 11, 12
L_KNEE, R_KNEE = 13, 14
L_ANKLE, R_ANKLE = 15, 16
TARGET_FPS = 25.0


df_index = pd.read_csv("data/processed/pose_index.csv")
if "clip_id" not in df_index.columns:
    df_index["clip_id"] = df_index["pose_path"].apply(lambda p: Path(p).stem)


def remuestrear_secuencia(kpts_xy: np.ndarray, fps_orig: float, fps_target: float = 25.0) -> np.ndarray:
    if np.isclose(fps_orig, fps_target, atol=0.5):
        return kpts_xy.astype(np.float32)
    T, J, C = kpts_xy.shape
    duracion = (T - 1) / fps_orig
    T_new = int(np.floor(duracion * fps_target)) + 1
    t_orig = np.linspace(0, duracion, T)
    t_new = np.linspace(0, duracion, T_new)
    out = np.full((T_new, J, C), np.nan, dtype=np.float32)
    for j in range(J):
        for c in range(C):
            y = kpts_xy[:, j, c]
            valid = ~np.isnan(y)
            if valid.sum() >= 2:
                y_res = np.interp(t_new, t_orig[valid], y[valid])
                mask_nan = np.interp(t_new, t_orig, np.isnan(y).astype(float)) > 0.5
                y_res[mask_nan] = np.nan
                out[:, j, c] = y_res
    return out


def interpolar_cortos(kpts_xy: np.ndarray, max_gap: int = 8) -> np.ndarray:
    out = kpts_xy.copy()
    T, J, C = out.shape
    t = np.arange(T)
    for j in range(J):
        for c in range(C):
            y = out[:, j, c]
            valid = ~np.isnan(y)
            if valid.sum() < 2:
                continue
            isnan = ~valid
            y_interp = np.interp(t, t[valid], y[valid])
            idx_change = np.where(np.diff(isnan.astype(int)) != 0)[0] + 1
            for seg in np.split(np.arange(T), idx_change):
                if isnan[seg[0]] and len(seg) <= max_gap:
                    out[seg, j, c] = y_interp[seg]
    return out


def normalizar_esqueleto(kpts_xy: np.ndarray) -> np.ndarray:
    mid_hip = 0.5 * (kpts_xy[:, L_HIP, :] + kpts_xy[:, R_HIP, :])
    mid_shoulder = 0.5 * (kpts_xy[:, L_SHOULDER, :] + kpts_xy[:, R_SHOULDER, :])
    torso_len = np.linalg.norm(mid_shoulder - mid_hip, axis=1)
    escala_fb = np.nanmedian(torso_len)
    if np.isnan(escala_fb) or escala_fb < 1e-3:
        escala_fb = 1.0
    torso_len = np.where(np.isnan(torso_len) | (torso_len < 1e-3), escala_fb, torso_len)
    return ((kpts_xy - mid_hip[:, None, :]) / torso_len[:, None, None]).astype(np.float32)


def obtener_ventanas_crudas(npz_path: str, win_sec: float = 2.5, step_sec: float = 0.5) -> np.ndarray:
    """Devuelve array (N, 63, 17, 2) de ventanas normalizadas sin NaN."""
    with np.load(npz_path) as data:
        kpts_xy = data["kpts_xy"]
        fps_orig = float(data["fps"])

    kpts = remuestrear_secuencia(kpts_xy, fps_orig, TARGET_FPS)
    kpts = interpolar_cortos(kpts, max_gap=int(0.32 * TARGET_FPS))
    kpts = normalizar_esqueleto(kpts)

    W = int(round(win_sec * TARGET_FPS))  # 63 frames
    S = int(round(step_sec * TARGET_FPS)) # 12 frames
    ventanas = []

    for start in range(0, len(kpts) - W + 1, S):
        win = kpts[start : start + W]
        req = win[:, [L_HIP, R_HIP, L_ANKLE, R_ANKLE], :]
        valid_frames = ~np.isnan(req).any(axis=(1, 2))
        if valid_frames.mean() < 0.85:
            continue
        if not valid_frames.all():
            win = pd.DataFrame(win.reshape(W, -1)).ffill().bfill().to_numpy().reshape(W, 17, 2)
        if not np.isnan(win).any():
            ventanas.append(win)

    return np.array(ventanas, dtype=np.float32) if ventanas else np.empty((0, W, 17, 2), dtype=np.float32)


# Extraer ventanas agrupadas por clip
clips_data = {}
meta_clips = []
for _, row in df_index.iterrows():
    wins = obtener_ventanas_crudas(row["pose_path"])
    if len(wins) == 0:
        continue
    cid = row["clip_id"]
    clips_data[cid] = wins
    meta_clips.append({
        "clip_id": cid,
        "dataset": row["dataset"],
        "clase_video": row["clase_video"],
        "estrato": f"{row['dataset']}_{row['clase_video']}",
        "n_win": len(wins),
    })

df_clips = pd.DataFrame(meta_clips)

# Separar clips supervisados (gavd/normal + urfall/adl + urfall/fall) de gavd/parkinsons
df_sup = df_clips[df_clips["estrato"] != "gavd_parkinsons"].copy()
df_pk = df_clips[df_clips["estrato"] == "gavd_parkinsons"].copy()

train_clips, temp_clips = train_test_split(
    df_sup, test_size=0.30, stratify=df_sup["estrato"], random_state=SEED
)
val_clips, test_clips = train_test_split(
    temp_clips, test_size=0.50, stratify=temp_clips["estrato"], random_state=SEED
)

print("Clips por partición:", len(train_clips), len(val_clips), len(test_clips))

Clips por partición: 32 7 8


In [6]:
LOWER_JOINTS = [L_KNEE, R_KNEE, L_ANKLE, R_ANKLE]
HIP_JOINTS = [L_HIP, R_HIP]


def simular_fog_acinetico(win: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Congela el avance de piernas desde t_onset manteniendo jitter realista de pose."""
    out = win.copy()
    W = len(out)
    # 70% transición dentro de la ventana, 30% bloqueo durante toda la ventana
    t_onset = int(rng.integers(8, W // 2)) if rng.random() < 0.7 else 0
    dur = W - t_onset

    # Congelar rodillas y tobillos en la postura de t_onset + leve oscilación postural
    base_lower = out[t_onset : t_onset + 1, LOWER_JOINTS, :]
    jitter = rng.normal(0.0, rng.uniform(0.003, 0.007), size=(dur, len(LOWER_JOINTS), 2))
    out[t_onset:, LOWER_JOINTS, :] = base_lower + jitter

    # Atenuar movimiento de caderas (85% reducción)
    base_hips = out[t_onset : t_onset + 1, HIP_JOINTS, :]
    out[t_onset:, HIP_JOINTS, :] = base_hips + 0.15 * (out[t_onset:, HIP_JOINTS, :] - base_hips)
    return out.astype(np.float32)


def simular_fog_temblor(win: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Congela locomoción e inyecta oscilación coherente de 3.5-7.5 Hz en rodillas y tobillos."""
    out = simular_fog_acinetico(win, rng)
    W = len(out)
    t_onset = int(rng.integers(6, W // 2)) if rng.random() < 0.7 else 0
    dur = W - t_onset

    t = np.arange(dur, dtype=np.float32) / TARGET_FPS
    freq = rng.uniform(3.5, 7.5)  # Banda clínica FoG (3-8 Hz)
    fase = rng.uniform(0, 2 * np.pi)
    # Ligero desfase entre pierna izquierda y derecha
    fase_r = fase + rng.uniform(-0.6, 0.6)

    amp_y = rng.uniform(0.015, 0.042)  # Amplitud vertical en torsos
    amp_x = amp_y * rng.uniform(0.25, 0.85)  # Cubre vistas sagital y frontal

    osc_l_y = amp_y * np.sin(2 * np.pi * freq * t + fase)
    osc_r_y = amp_y * np.sin(2 * np.pi * freq * t + fase_r)
    osc_l_x = amp_x * np.cos(2 * np.pi * freq * t + fase)
    osc_r_x = amp_x * np.cos(2 * np.pi * freq * t + fase_r)

    # Tobillos reciben 100% de la amplitud; rodillas ~60%
    out[t_onset:, L_ANKLE, 1] += osc_l_y
    out[t_onset:, R_ANKLE, 1] += osc_r_y
    out[t_onset:, L_ANKLE, 0] += osc_l_x
    out[t_onset:, R_ANKLE, 0] += osc_r_x

    out[t_onset:, L_KNEE, 1] += 0.6 * osc_l_y
    out[t_onset:, R_KNEE, 1] += 0.6 * osc_r_y
    out[t_onset:, L_KNEE, 0] += 0.6 * osc_l_x
    out[t_onset:, R_KNEE, 0] += 0.6 * osc_r_x

    return out.astype(np.float32)


def construir_particion(df_sub: pd.DataFrame, rng: np.random.Generator):
    """
    Clase 0 (No FoG): gavd/normal original + urfall/adl (paradas/sentarse) + urfall/fall
    Clase 1 (FoG): bloqueos sintéticos (acinéticos + temblor) sobre gavd/normal
    """
    X_wins, y_labels, subtipos, clip_ids = [], [], [], []

    for _, row in df_sub.iterrows():
        cid = row["clip_id"]
        estrato = row["estrato"]
        wins = clips_data[cid]

        for w in wins:
            # Siempre añadir la ventana real como clase negativa (0)
            X_wins.append(w)
            y_labels.append(0)
            subtipos.append(estrato)
            clip_ids.append(cid)

            # Si es marcha normal, generar pares sintéticos positivos (1)
            if estrato == "gavd_normal":
                X_wins.append(simular_fog_acinetico(w, rng))
                y_labels.append(1)
                subtipos.append("fog_acinetico")
                clip_ids.append(cid)

                X_wins.append(simular_fog_temblor(w, rng))
                y_labels.append(1)
                subtipos.append("fog_temblor")
                clip_ids.append(cid)

    return np.array(X_wins, dtype=np.float32), np.array(y_labels, dtype=np.int64), np.array(subtipos), np.array(clip_ids)


X_tr_raw, y_tr, sub_tr, cid_tr = construir_particion(train_clips, rng)
X_va_raw, y_va, sub_va, cid_va = construir_particion(val_clips, rng)
X_te_raw, y_te, sub_te, cid_te = construir_particion(test_clips, rng)

print(f"Train: {len(y_tr)} ventanas (FoG={y_tr.mean():.1%}) | Val: {len(y_va)} | Test: {len(y_te)}")

Train: 362 ventanas (FoG=33.7%) | Val: 49 | Test: 292


In [7]:
import torch
from torch.utils.data import TensorDataset, DataLoader

JOINTS_POS = [L_SHOULDER, R_SHOULDER, L_HIP, R_HIP, L_KNEE, R_KNEE, L_ANKLE, R_ANKLE]  # 8 -> 16 cols
JOINTS_VEL = [L_HIP, R_HIP, L_KNEE, R_KNEE, L_ANKLE, R_ANKLE]                          # 6 -> 12 cols


def angulo_articulacion(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> np.ndarray:
    """Ángulo en radianes en el vértice b formado por los segmentos ba y bc. Shape: (N, W)."""
    ba = a - b
    bc = c - b
    num = np.sum(ba * bc, axis=-1)
    den = np.linalg.norm(ba, axis=-1) * np.linalg.norm(bc, axis=-1) + 1e-6
    return np.arccos(np.clip(num / den, -1.0, 1.0))


def extraer_tensores_secuenciales(X_raw: np.ndarray, fps: float = 25.0) -> np.ndarray:
    """Convierte (N, W, 17, 2) -> (N, W, 32)."""
    N, W, _, _ = X_raw.shape

    # 1. Posiciones normalizadas (N, W, 16)
    pos = X_raw[:, :, JOINTS_POS, :].reshape(N, W, -1)

    # 2. Velocidades instantáneas (N, W, 12)
    vel = np.diff(X_raw[:, :, JOINTS_VEL, :], axis=1, prepend=X_raw[:, :1, JOINTS_VEL, :]) * fps
    vel = vel.reshape(N, W, -1)

    # 3. Ángulos de rodilla izq/der y altura relativa tobillo-cadera (N, W, 4)
    ang_l = angulo_articulacion(X_raw[:, :, L_HIP], X_raw[:, :, L_KNEE], X_raw[:, :, L_ANKLE])
    ang_r = angulo_articulacion(X_raw[:, :, R_HIP], X_raw[:, :, R_KNEE], X_raw[:, :, R_ANKLE])
    dy_l = X_raw[:, :, L_ANKLE, 1] - X_raw[:, :, L_HIP, 1]
    dy_r = X_raw[:, :, R_ANKLE, 1] - X_raw[:, :, R_HIP, 1]
    geom = np.stack([ang_l, ang_r, dy_l, dy_r], axis=-1)

    return np.concatenate([pos, vel, geom], axis=-1).astype(np.float32)


X_tr = extraer_tensores_secuenciales(X_tr_raw)
X_va = extraer_tensores_secuenciales(X_va_raw)
X_te = extraer_tensores_secuenciales(X_te_raw)

# Estandarización por canal usando exclusivamente TRAIN
mean_ch = X_tr.mean(axis=(0, 1), keepdims=True)
std_ch = X_tr.std(axis=(0, 1), keepdims=True) + 1e-6

X_tr_norm = (X_tr - mean_ch) / std_ch
X_va_norm = (X_va - mean_ch) / std_ch
X_te_norm = (X_te - mean_ch) / std_ch

train_loader = DataLoader(TensorDataset(torch.tensor(X_tr_norm), torch.tensor(y_tr, dtype=torch.float32)), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_va_norm), torch.tensor(y_va, dtype=torch.float32)), batch_size=64)
test_loader = DataLoader(TensorDataset(torch.tensor(X_te_norm), torch.tensor(y_te, dtype=torch.float32)), batch_size=64)

In [8]:
import torch.nn as nn
from sklearn.metrics import classification_report, roc_auc_score

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Entrenando en:", device)


class CNN1D_BiGRU(nn.Module):
    def __init__(self, in_channels: int = 32, hidden_dim: int = 48):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.GELU(),
        )
        self.gru = nn.GRU(64, hidden_dim, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 4, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, W, C) -> (B, C, W)
        h = self.conv(x.transpose(1, 2)).transpose(1, 2)
        h, _ = self.gru(h)  # (B, W, 2*hidden_dim)
        # Pooling global promedio + máximo sobre el tiempo
        h_pool = torch.cat([h.mean(dim=1), h.max(dim=1).values], dim=-1)
        return self.head(h_pool).squeeze(-1)


model = CNN1D_BiGRU(in_channels=X_tr_norm.shape[-1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

best_val_loss = float("inf")
best_state = None

for epoch in range(1, 36):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_loss += criterion(model(xb), yb).item() * len(yb)
    val_loss /= len(y_va)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
model.eval()

# Evaluación en Test
with torch.no_grad():
    probs_te = torch.sigmoid(model(torch.tensor(X_te_norm, device=device))).cpu().numpy()
preds_te = (probs_te >= 0.5).astype(int)

print("\n=== RESULTADOS EN TEST (SPLIT POR CLIP) ===")
print(classification_report(y_te, preds_te, target_names=["No FoG", "FoG"], digits=3))
print(f"ROC-AUC Test: {roc_auc_score(y_te, probs_te):.4f}")

# Auditoría por subtipo (comparativa directa contra el 17.94% FPR del baseline en urfall/adl)
df_eval = pd.DataFrame({"subtipo": sub_te, "y_true": y_te, "prob_fog": probs_te, "pred": preds_te})
print("\nTasa de detección positiva por subtipo en Test (%):")
print(df_eval.groupby("subtipo")["pred"].mean().mul(100).round(2))

# Inferencia sobre clips reales de gavd/parkinsons
X_pk_raw = np.concatenate([clips_data[c] for c in df_pk["clip_id"]], axis=0)
X_pk_norm = (extraer_tensores_secuenciales(X_pk_raw) - mean_ch) / std_ch
with torch.no_grad():
    probs_pk = torch.sigmoid(model(torch.tensor(X_pk_norm, device=device))).cpu().numpy()

print(f"\nVentanas detectadas como FoG en gavd/parkinsons real: {(probs_pk >= 0.5).mean()*100:.2f}% (prob media={probs_pk.mean():.3f})")

Entrenando en: cuda

=== RESULTADOS EN TEST (SPLIT POR CLIP) ===
              precision    recall  f1-score   support

      No FoG      1.000     0.425     0.597       134
         FoG      0.672     1.000     0.804       158

    accuracy                          0.736       292
   macro avg      0.836     0.713     0.700       292
weighted avg      0.823     0.736     0.709       292

ROC-AUC Test: 0.8597

Tasa de detección positiva por subtipo en Test (%):
subtipo
fog_acinetico    100.00
fog_temblor      100.00
gavd_normal       92.41
urfall_adl         7.69
urfall_fall        6.25
Name: pred, dtype: float64

Ventanas detectadas como FoG en gavd/parkinsons real: 77.16% (prob media=0.749)


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, roc_auc_score

# Solo tren inferior: caderas, rodillas y tobillos (6 articulaciones -> 28 canales)
JOINTS_POS = [L_HIP, R_HIP, L_KNEE, R_KNEE, L_ANKLE, R_ANKLE]  # 12 cols
JOINTS_VEL = [L_HIP, R_HIP, L_KNEE, R_KNEE, L_ANKLE, R_ANKLE]  # 12 cols


def extraer_canales_piernas(X_raw: np.ndarray, fps: float = 25.0) -> np.ndarray:
    """Convierte (N, W, 17, 2) -> (N, W, 28) usando solo tren inferior."""
    N, W, _, _ = X_raw.shape
    pos = X_raw[:, :, JOINTS_POS, :].reshape(N, W, -1)
    vel = np.diff(X_raw[:, :, JOINTS_VEL, :], axis=1, prepend=X_raw[:, :1, JOINTS_VEL, :]) * fps
    vel = vel.reshape(N, W, -1)

    ang_l = angulo_articulacion(X_raw[:, :, L_HIP], X_raw[:, :, L_KNEE], X_raw[:, :, L_ANKLE])
    ang_r = angulo_articulacion(X_raw[:, :, R_HIP], X_raw[:, :, R_KNEE], X_raw[:, :, R_ANKLE])
    dy_l = X_raw[:, :, L_ANKLE, 1] - X_raw[:, :, L_HIP, 1]
    dy_r = X_raw[:, :, R_ANKLE, 1] - X_raw[:, :, R_HIP, 1]
    geom = np.stack([ang_l, ang_r, dy_l, dy_r], axis=-1)

    return np.concatenate([pos, vel, geom], axis=-1).astype(np.float32)


def construir_dataset_equilibrado(df_sub: pd.DataFrame, rng: np.random.Generator):
    """Genera 1 positivo sintético (acinético O temblor) por cada ventana gavd_normal -> GAVD 50/50."""
    X_wins, y_labels, subtipos, clip_ids = [], [], [], []

    for _, row in df_sub.iterrows():
        cid = row["clip_id"]
        estrato = row["estrato"]
        for w in clips_data[cid]:
            # Ventana real -> Clase 0
            X_wins.append(w)
            y_labels.append(0)
            subtipos.append(estrato)
            clip_ids.append(cid)

            # Si es marcha normal, generar 1 bloqueo sintético -> Clase 1
            if estrato == "gavd_normal":
                if rng.random() < 0.5:
                    X_wins.append(simular_fog_acinetico(w, rng))
                    subtipos.append("fog_acinetico")
                else:
                    X_wins.append(simular_fog_temblor(w, rng))
                    subtipos.append("fog_temblor")
                y_labels.append(1)
                clip_ids.append(cid)

    return (
        np.array(X_wins, dtype=np.float32),
        np.array(y_labels, dtype=np.int64),
        np.array(subtipos),
        np.array(clip_ids),
    )


class CNN1D_Compacta(nn.Module):
    def __init__(self, in_channels: int = 28):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.35),
            nn.Conv1d(32, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.35),
        )
        self.head = nn.Linear(64, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.net(x.transpose(1, 2))  # (B, 32, W)
        h_pool = torch.cat([h.mean(dim=-1), h.max(dim=-1).values], dim=-1)
        return self.head(h_pool).squeeze(-1)


# Construir dataset completo supervisado y evaluar con 5-Fold agrupado por clip
rng_cv = np.random.default_rng(SEED)
X_all_raw, y_all, sub_all, cid_all = construir_dataset_equilibrado(df_sup, rng_cv)
X_all = extraer_canales_piernas(X_all_raw)

sgkf = StratifiedGroupKFold(n_splits=5)
oof_probs = np.zeros(len(y_all), dtype=np.float32)

for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X_all, y_all, groups=cid_all), 1):
    mean_f = X_all[tr_idx].mean(axis=(0, 1), keepdims=True)
    std_f = X_all[tr_idx].std(axis=(0, 1), keepdims=True) + 1e-6

    X_tr_f = (X_all[tr_idx] - mean_f) / std_f
    X_te_f = (X_all[te_idx] - mean_f) / std_f

    loader_tr = DataLoader(
        TensorDataset(torch.tensor(X_tr_f), torch.tensor(y_all[tr_idx], dtype=torch.float32)),
        batch_size=32,
        shuffle=True,
    )

    torch.manual_seed(SEED + fold)
    net = CNN1D_Compacta(in_channels=X_all.shape[-1]).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=3e-2)
    loss_fn = nn.BCEWithLogitsLoss()

    net.train()
    for _ in range(30):
        for xb, yb in loader_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(net(xb), yb)
            loss.backward()
            opt.step()

    net.eval()
    with torch.no_grad():
        oof_probs[te_idx] = torch.sigmoid(net(torch.tensor(X_te_f, device=device))).cpu().numpy()

oof_preds = (oof_probs >= 0.5).astype(int)
print("=== RESULTADOS GLOBALES 5-FOLD (SIN FUGA ENTRE CLIPS) ===")
print(classification_report(y_all, oof_preds, target_names=["No FoG", "FoG"], digits=3))
print(f"ROC-AUC OOF: {roc_auc_score(y_all, oof_probs):.4f}")

df_oof = pd.DataFrame({"subtipo": sub_all, "pred": oof_preds})
print("\nTasa de detección positiva por subtipo (%):")
print(df_oof.groupby("subtipo")["pred"].mean().mul(100).round(2))

=== RESULTADOS GLOBALES 5-FOLD (SIN FUGA ENTRE CLIPS) ===
              precision    recall  f1-score   support

      No FoG      0.947     0.814     0.876       415
         FoG      0.619     0.868     0.723       144

    accuracy                          0.828       559
   macro avg      0.783     0.841     0.799       559
weighted avg      0.862     0.828     0.836       559

ROC-AUC OOF: 0.9011

Tasa de detección positiva por subtipo (%):
subtipo
fog_acinetico    80.30
fog_temblor      92.31
gavd_normal      52.78
urfall_adl        0.00
urfall_fall       2.08
Name: pred, dtype: float64


In [10]:
from sklearn.metrics import f1_score

JOINTS_KIN = [L_HIP, R_HIP, L_KNEE, R_KNEE, L_ANKLE, R_ANKLE]  # 6 articulaciones


def extraer_cinematica_pura(X_raw: np.ndarray, fps: float = 25.0) -> np.ndarray:
    """
    Extrae 30 canales invariantes a la vista de cámara:
    velocidades (12) + aceleraciones (12) + ángulos de rodilla, vel. angular y altura tobillo-cadera (6).
    """
    vel = np.diff(X_raw[:, :, JOINTS_KIN, :], axis=1, prepend=X_raw[:, :1, JOINTS_KIN, :]) * fps
    acc = np.diff(vel, axis=1, prepend=vel[:, :1, :, :]) * fps

    vel_flat = vel.reshape(len(X_raw), X_raw.shape[1], -1)
    acc_flat = acc.reshape(len(X_raw), X_raw.shape[1], -1)

    ang_l = angulo_articulacion(X_raw[:, :, L_HIP], X_raw[:, :, L_KNEE], X_raw[:, :, L_ANKLE])
    ang_r = angulo_articulacion(X_raw[:, :, R_HIP], X_raw[:, :, R_KNEE], X_raw[:, :, R_ANKLE])
    dang_l = np.diff(ang_l, axis=1, prepend=ang_l[:, :1]) * fps
    dang_r = np.diff(ang_r, axis=1, prepend=ang_r[:, :1]) * fps
    dy_l = X_raw[:, :, L_ANKLE, 1] - X_raw[:, :, L_HIP, 1]
    dy_r = X_raw[:, :, R_ANKLE, 1] - X_raw[:, :, R_HIP, 1]

    geom = np.stack([ang_l, ang_r, dang_l, dang_r, dy_l, dy_r], axis=-1)
    return np.concatenate([vel_flat, acc_flat, geom], axis=-1).astype(np.float32)


X_kin = extraer_cinematica_pura(X_all_raw)

# Dar peso 2.0 a gavd_normal para que el modelo no lo confunda con FoG
pesos_muestra = np.where(sub_all == "gavd_normal", 2.0, 1.0).astype(np.float32)

sgkf = StratifiedGroupKFold(n_splits=5)
oof_probs_kin = np.zeros(len(y_all), dtype=np.float32)
modelos_fold = []

for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X_kin, y_all, groups=cid_all), 1):
    mean_f = X_kin[tr_idx].mean(axis=(0, 1), keepdims=True)
    std_f = X_kin[tr_idx].std(axis=(0, 1), keepdims=True) + 1e-6

    X_tr_f = (X_kin[tr_idx] - mean_f) / std_f
    X_te_f = (X_kin[te_idx] - mean_f) / std_f

    ds_tr = TensorDataset(
        torch.tensor(X_tr_f),
        torch.tensor(y_all[tr_idx], dtype=torch.float32),
        torch.tensor(pesos_muestra[tr_idx]),
    )
    loader_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)

    torch.manual_seed(SEED + fold)
    net = CNN1D_Compacta(in_channels=X_kin.shape[-1]).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=3e-2)
    loss_fn = nn.BCEWithLogitsLoss(reduction="none")

    net.train()
    for _ in range(30):
        for xb, yb, wb in loader_tr:
            xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
            opt.zero_grad()
            loss = (loss_fn(net(xb), yb) * wb).mean()
            loss.backward()
            opt.step()

    net.eval()
    with torch.no_grad():
        oof_probs_kin[te_idx] = torch.sigmoid(net(torch.tensor(X_te_f, device=device))).cpu().numpy()
    modelos_fold.append((net, mean_f, std_f))

# Buscar el umbral óptimo que maximiza el F1-score global
umbrales = np.linspace(0.30, 0.80, 51)
f1s = [f1_score(y_all, (oof_probs_kin >= u).astype(int)) for u in umbrales]
best_thr = float(umbrales[int(np.argmax(f1s))])
oof_preds_opt = (oof_probs_kin >= best_thr).astype(int)

print(f"=== RESULTADOS CINEMÁTICA PURA (Umbral óptimo = {best_thr:.2f}) ===")
print(classification_report(y_all, oof_preds_opt, target_names=["No FoG", "FoG"], digits=3))
print(f"ROC-AUC OOF: {roc_auc_score(y_all, oof_probs_kin):.4f}")

df_oof_kin = pd.DataFrame({"subtipo": sub_all, "prob": oof_probs_kin, "pred": oof_preds_opt})
print("\nProbabilidad media y tasa de detección por subtipo:")
print(df_oof_kin.groupby("subtipo").agg(prob_media=("prob", "mean"), deteccion_pct=("pred", lambda s: s.mean() * 100)).round(2))

# Inferencia ensemble (media de los 5 folds) sobre los clips reales de gavd/parkinsons
X_pk_raw = np.concatenate([clips_data[c] for c in df_pk["clip_id"]], axis=0)
X_pk_kin = extraer_cinematica_pura(X_pk_raw)

probs_pk_folds = []
for net, mean_f, std_f in modelos_fold:
    X_pk_norm = (X_pk_kin - mean_f) / std_f
    with torch.no_grad():
        probs_pk_folds.append(torch.sigmoid(net(torch.tensor(X_pk_norm, device=device))).cpu().numpy())

probs_pk_ens = np.mean(probs_pk_folds, axis=0)
print(f"\nDetección en gavd/parkinsons real (359 ventanas): {(probs_pk_ens >= best_thr).mean()*100:.2f}% (prob media={probs_pk_ens.mean():.3f})")

=== RESULTADOS CINEMÁTICA PURA (Umbral óptimo = 0.39) ===
              precision    recall  f1-score   support

      No FoG      0.955     0.766     0.850       415
         FoG      0.571     0.896     0.697       144

    accuracy                          0.800       559
   macro avg      0.763     0.831     0.774       559
weighted avg      0.856     0.800     0.811       559

ROC-AUC OOF: 0.8656

Probabilidad media y tasa de detección por subtipo:
               prob_media  deteccion_pct
subtipo                                 
fog_acinetico        0.54          90.91
fog_temblor          0.63          88.46
gavd_normal          0.40          53.47
urfall_adl           0.18           8.97
urfall_fall          0.10           0.00

Detección en gavd/parkinsons real (359 ventanas): 27.02% (prob media=0.345)


In [11]:
JOINTS_POS = [L_HIP, R_HIP, L_KNEE, R_KNEE, L_ANKLE, R_ANKLE]
JOINTS_VEL = [L_HIP, R_HIP, L_KNEE, R_KNEE, L_ANKLE, R_ANKLE]


def suavizar_temporal(X_raw: np.ndarray) -> np.ndarray:
    """Filtro suave [0.25, 0.5, 0.25] para reducir jitter de pose sin perder 3-8 Hz."""
    X_pad = np.pad(X_raw, ((0, 0), (1, 1), (0, 0), (0, 0)), mode="edge")
    return (0.25 * X_pad[:, :-2] + 0.50 * X_pad[:, 1:-1] + 0.25 * X_pad[:, 2:]).astype(np.float32)


def extraer_features_finales(X_raw: np.ndarray, fps: float = 25.0) -> np.ndarray:
    X_s = suavizar_temporal(X_raw)
    N, W, _, _ = X_s.shape

    pos = X_s[:, :, JOINTS_POS, :].reshape(N, W, -1)
    vel = (np.diff(X_s[:, :, JOINTS_VEL, :], axis=1, prepend=X_s[:, :1, JOINTS_VEL, :]) * fps).reshape(N, W, -1)

    ang_l = angulo_articulacion(X_s[:, :, L_HIP], X_s[:, :, L_KNEE], X_s[:, :, L_ANKLE])
    ang_r = angulo_articulacion(X_s[:, :, R_HIP], X_s[:, :, R_KNEE], X_s[:, :, R_ANKLE])
    dy_l = X_s[:, :, L_ANKLE, 1] - X_s[:, :, L_HIP, 1]
    dy_r = X_s[:, :, R_ANKLE, 1] - X_s[:, :, R_HIP, 1]
    geom = np.stack([ang_l, ang_r, dy_l, dy_r], axis=-1)

    return np.concatenate([pos, vel, geom], axis=-1).astype(np.float32)


X_final = extraer_features_finales(X_all_raw)
pesos_muestra = np.where(sub_all == "gavd_normal", 2.0, 1.0).astype(np.float32)

sgkf = StratifiedGroupKFold(n_splits=5)
oof_probs_fin = np.zeros(len(y_all), dtype=np.float32)
modelos_finales = []

for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X_final, y_all, groups=cid_all), 1):
    mean_f = X_final[tr_idx].mean(axis=(0, 1), keepdims=True)
    std_f = X_final[tr_idx].std(axis=(0, 1), keepdims=True) + 1e-6

    X_tr_f = (X_final[tr_idx] - mean_f) / std_f
    X_te_f = (X_final[te_idx] - mean_f) / std_f

    loader_tr = DataLoader(
        TensorDataset(
            torch.tensor(X_tr_f),
            torch.tensor(y_all[tr_idx], dtype=torch.float32),
            torch.tensor(pesos_muestra[tr_idx]),
        ),
        batch_size=32,
        shuffle=True,
    )

    torch.manual_seed(SEED + fold)
    net = CNN1D_Compacta(in_channels=X_final.shape[-1]).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=3e-2)
    loss_fn = nn.BCEWithLogitsLoss(reduction="none")

    net.train()
    for _ in range(30):
        for xb, yb, wb in loader_tr:
            xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
            opt.zero_grad()
            loss = (loss_fn(net(xb), yb) * wb).mean()
            loss.backward()
            opt.step()

    net.eval()
    with torch.no_grad():
        oof_probs_fin[te_idx] = torch.sigmoid(net(torch.tensor(X_te_f, device=device))).cpu().numpy()
    modelos_finales.append((net, mean_f, std_f))

# Calibrar umbral al 10% de FPR en marcha sana (igualando el 10.42% del baseline)
thr_10fpr = float(np.quantile(oof_probs_fin[sub_all == "gavd_normal"], 0.90))
oof_preds_fin = (oof_probs_fin >= thr_10fpr).astype(int)

print(f"=== MODELO FINAL CNN1D (Umbral calibrado al 10% FPR sano = {thr_10fpr:.3f}) ===")
print(classification_report(y_all, oof_preds_fin, target_names=["No FoG", "FoG"], digits=3))
print(f"ROC-AUC OOF: {roc_auc_score(y_all, oof_probs_fin):.4f}")

df_res = pd.DataFrame({"subtipo": sub_all, "prob": oof_probs_fin, "pred": oof_preds_fin})
print("\nTasa de detección por subtipo (%):")
print(df_res.groupby("subtipo").agg(prob_media=("prob", "mean"), deteccion_pct=("pred", lambda s: s.mean() * 100)).round(2))

# Inferencia sobre pacientes reales de gavd/parkinsons desglosada por clip
filas_pk = []
for cid in df_pk["clip_id"]:
    X_c = extraer_features_finales(clips_data[cid])
    probs_c = np.mean([
        torch.sigmoid(net(torch.tensor((X_c - m) / s, device=device))).cpu().numpy()
        for net, m, s in modelos_finales
    ], axis=0)
    filas_pk.append({
        "clip_id": cid,
        "n_ventanas": len(probs_c),
        "prob_media": float(probs_c.mean()),
        "prob_max": float(probs_c.max()),
        "pct_fog": float((probs_c >= thr_10fpr).mean() * 100),
    })

df_pk_res = pd.DataFrame(filas_pk).sort_values("pct_fog", ascending=False)
print(f"\nDetección global en gavd/parkinsons real: {(df_pk_res['pct_fog'] * df_pk_res['n_ventanas']).sum() / df_pk_res['n_ventanas'].sum():.2f}%")
print("\nTop 8 clips de gavd/parkinsons con mayor porcentaje de bloqueo detectado:")
print(df_pk_res.head(8).to_string(index=False))

=== MODELO FINAL CNN1D (Umbral calibrado al 10% FPR sano = 0.835) ===
              precision    recall  f1-score   support

      No FoG      0.792     0.964     0.870       415
         FoG      0.722     0.271     0.394       144

    accuracy                          0.785       559
   macro avg      0.757     0.617     0.632       559
weighted avg      0.774     0.785     0.747       559

ROC-AUC OOF: 0.8947

Tasa de detección por subtipo (%):
               prob_media  deteccion_pct
subtipo                                 
fog_acinetico        0.66          31.82
fog_temblor          0.71          23.08
gavd_normal          0.46          10.42
urfall_adl           0.16           0.00
urfall_fall          0.11           0.00


RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.